# 02 Cleaning Notebook

This notebook turns the raw Online Retail II file into a clean analysis dataset. The goal is to keep the process transparent: every step shows what changed, why we changed it, and how many rows were affected. That makes the cleaning process easy to defend in the report and in the viva.

The repository currently stores the raw data as CSV, so the loader below can read the CSV/ZIP version while still supporting the original Excel workbook if it is restored later.

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
print('Libraries imported successfully: pandas, numpy, and warnings.')

Libraries imported successfully: pandas, numpy, and warnings.


## Step 2: Load the raw dataset

We load the raw file into `df` at the start of the cleaning notebook. This gives us one dataframe that all later steps can transform in sequence. The loader is flexible so it can read the CSV version that exists in the repo or the original Excel workbook if that version becomes available later.

In [2]:
def load_raw_dataset() -> pd.DataFrame:
    """Load the raw Online Retail II data from Excel, CSV, or ZIP."""
    candidates = [
        Path('data/raw/online_retail_II.xlsx'),
        Path('data/raw/online_retail_II.csv'),
        Path('data/raw/online_retail_II.zip'),
        Path('../data/raw/online_retail_II.xlsx'),
        Path('../data/raw/online_retail_II.csv'),
        Path('../data/raw/online_retail_II.zip'),
    ]

    for path in candidates:
        if not path.exists():
            continue

        if path.suffix.lower() in {'.xlsx', '.xls'}:
            workbook = pd.ExcelFile(path)
            sheet_names = workbook.sheet_names[:2]
            frames = [pd.read_excel(path, sheet_name=sheet) for sheet in sheet_names]
            print(f'Loaded workbook from: {path}')
            print(f'Sheets read: {sheet_names}')
            return pd.concat(frames, ignore_index=True)

        if path.suffix.lower() == '.zip':
            import zipfile
            with zipfile.ZipFile(path) as archive:
                csv_members = [name for name in archive.namelist() if name.lower().endswith('.csv')]
                if not csv_members:
                    raise FileNotFoundError('The ZIP archive does not contain a CSV file.')
                with archive.open(csv_members[0]) as file_obj:
                    print(f'Loaded CSV from ZIP archive: {path}')
                    return pd.read_csv(file_obj)

        print(f'Loaded CSV file from: {path}')
        return pd.read_csv(path)

    raise FileNotFoundError('Could not find online_retail_II.xlsx, online_retail_II.csv, or online_retail_II.zip in data/raw/.')

df = load_raw_dataset()
print(f'Initial shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

Loaded CSV file from: ../data/raw/online_retail_II.csv
Initial shape: 1,067,371 rows x 8 columns


## Step 3: Rename columns to snake_case

We rename the raw columns into cleaner snake_case names because consistent column names make the rest of the pipeline easier to read, easier to debug, and easier to reuse in notebooks and scripts. This step does not change the number of rows, but it makes the dataframe much easier to work with.

In [3]:
rows_before = len(df)
print('Columns before rename:')
print(list(df.columns))

df = df.rename(columns={
    'Invoice': 'invoice_no',
    'StockCode': 'stock_code',
    'Description': 'description',
    'Quantity': 'quantity',
    'InvoiceDate': 'invoice_date',
    'Price': 'unit_price',
    'Customer ID': 'customer_id',
    'Country': 'country',
})

rows_after = len(df)
print('Columns after rename:')
print(list(df.columns))
print(f'Rows before: {rows_before:,} | Rows after: {rows_after:,} | Rows removed: {rows_before - rows_after:,}')

Columns before rename:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Columns after rename:
['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country']
Rows before: 1,067,371 | Rows after: 1,067,371 | Rows removed: 0


## Step 4: Parse `invoice_date` to datetime

We convert the invoice date column into a real datetime type so we can later sort transactions, group by time period, and calculate recency for RFM analysis. The notebook prints the dtype before and after the conversion, plus the minimum and maximum dates found in the data.

In [4]:
rows_before = len(df)
before_dtype = df['invoice_date'].dtype
df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')
after_dtype = df['invoice_date'].dtype
rows_after = len(df)

print(f'Dtype before parsing: {before_dtype}')
print(f'Dtype after parsing: {after_dtype}')
print(f'Min invoice_date: {df["invoice_date"].min()}')
print(f'Max invoice_date: {df["invoice_date"].max()}')
print(f'Rows before: {rows_before:,} | Rows after: {rows_after:,} | Rows removed: {rows_before - rows_after:,}')

Dtype before parsing: object
Dtype after parsing: datetime64[ns]
Min invoice_date: 2009-12-01 07:45:00
Max invoice_date: 2011-12-09 12:50:00
Rows before: 1,067,371 | Rows after: 1,067,371 | Rows removed: 0


## Step 5: Drop rows where `customer_id` is null

We remove rows with missing customer IDs because RFM segmentation needs a customer identifier for each transaction. Without that ID, the row cannot be assigned to a customer segment. This step will usually remove a large number of rows in this dataset, so we print the count before and after the filter.

In [5]:
rows_before = len(df)
null_mask = df['customer_id'].isna() | df['customer_id'].astype('string').str.strip().isin(['', 'nan', 'None'])
removed = int(null_mask.sum())
df = df.loc[~null_mask].copy()
rows_after = len(df)

print(f'Rows before: {rows_before:,}')
print(f'Rows after: {rows_after:,}')
print(f'Rows removed: {removed:,}')

Rows before: 1,067,371
Rows after: 824,364
Rows removed: 243,007


## Step 6: Convert `customer_id` to string and strip the `.0` suffix

After removing null IDs, we convert `customer_id` into a string so it behaves like an identifier instead of a number. Many Excel or CSV imports read IDs such as `13085.0`, so we strip the trailing `.0` to restore the customer ID to a clean text value. This step does not remove rows; it only standardizes the format.

In [ ]:
rows_before = len(df)
sample_before = df['customer_id'].head(5).tolist()
df['customer_id'] = df['customer_id'].astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
sample_after = df['customer_id'].head(5).tolist()
rows_after = len(df)

print('Sample customer_id values before:')
print(sample_before)
print('Sample customer_id values after:')
print(sample_after)
print(f'Rows before: {rows_before:,} | Rows after: {rows_after:,} | Rows removed: {rows_before - rows_after:,}')

## Step 7: Remove cancellations

Invoices starting with `C` are cancelled orders. They should not be treated as normal purchases because they would distort revenue, frequency, and retention metrics. We count the cancellations first, then remove them, and print the row counts before and after the filter.

In [ ]:
rows_before = len(df)
cancel_mask = df['invoice_no'].astype('string').str.startswith('C', na=False)
removed = int(cancel_mask.sum())
df = df.loc[~cancel_mask].copy()
rows_after = len(df)

print(f'Cancellation rows found: {removed:,}')
print(f'Rows before: {rows_before:,}')
print(f'Rows after: {rows_after:,}')
print(f'Rows removed: {removed:,}')

## Step 8: Remove rows where `quantity <= 0`

Negative or zero quantities are not valid purchase quantities for normal revenue analysis. They often represent returns or other edge cases. We convert the column to numeric, remove the invalid rows, and print how many rows were affected.

In [ ]:
rows_before = len(df)
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
invalid_mask = df['quantity'].le(0)
removed = int(invalid_mask.sum())
df = df.loc[~invalid_mask].copy()
rows_after = len(df)

print(f'Rows before: {rows_before:,}')
print(f'Rows after: {rows_after:,}')
print(f'Rows removed: {removed:,}')

## Step 9: Remove rows where `unit_price <= 0`

A price of zero or less is not useful for normal sales analysis. It can represent free items, special adjustments, or data problems. We remove these rows so the revenue calculation later is valid and easy to interpret.

In [ ]:
rows_before = len(df)
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
invalid_mask = df['unit_price'].le(0)
removed = int(invalid_mask.sum())
df = df.loc[~invalid_mask].copy()
rows_after = len(df)

print(f'Rows before: {rows_before:,}')
print(f'Rows after: {rows_after:,}')
print(f'Rows removed: {removed:,}')

## Step 10: Remove non-product stock codes

Some stock codes in the raw file are not real products. They represent postage, manual charges, discounts, or gift-related control lines. These records should not be treated as product sales, so we remove them using the approved non-product list.

In [ ]:
rows_before = len(df)
non_product_stock_codes = [
    'POST', 'D', 'C2', 'M', 'BANK CHARGES', 'AMAZONFEE',
    'DCGSSBOY', 'DCGSSGIRL', 'gift_0001_10', 'gift_0001_20',
    'gift_0001_30', 'gift_0001_40', 'gift_0001_50', 'DOT',
]
non_product_mask = df['stock_code'].astype('string').isin(non_product_stock_codes)
removed = int(non_product_mask.sum())
df = df.loc[~non_product_mask].copy()
rows_after = len(df)

print(f'Rows before: {rows_before:,}')
print(f'Rows after: {rows_after:,}')
print(f'Rows removed: {removed:,}')

## Step 11: Strip whitespace and uppercase `description` and `country`

Text standardization makes labels consistent for dashboards and grouping. We remove leading and trailing spaces and convert both columns to uppercase so that similar values do not appear as separate categories just because of formatting differences. This step should not remove rows.

In [ ]:
rows_before = len(df)
sample_before = df[['description', 'country']].head(3).copy()
df['description'] = df['description'].astype('string').str.strip().str.upper()
df['country'] = df['country'].astype('string').str.strip().str.upper()
sample_after = df[['description', 'country']].head(3).copy()
rows_after = len(df)

print('Sample values before text cleaning:')
print(sample_before.to_string(index=False))
print('Sample values after text cleaning:')
print(sample_after.to_string(index=False))
print(f'Rows before: {rows_before:,} | Rows after: {rows_after:,} | Rows removed: {rows_before - rows_after:,}')

## Step 12: Add the `revenue` column

Revenue is one of the most important derived fields in this project. We compute it as `quantity * unit_price` so that each line item contributes a monetary value we can later aggregate by customer, segment, country, or time period.

In [ ]:
rows_before = len(df)
df['revenue'] = df['quantity'] * df['unit_price']
rows_after = len(df)
total_revenue = df['revenue'].sum()

print(f'Total revenue: £{total_revenue:,.2f}')
print(df[['invoice_no', 'customer_id', 'quantity', 'unit_price', 'revenue']].head().to_string(index=False))
print(f'Rows before: {rows_before:,} | Rows after: {rows_after:,} | Rows removed: {rows_before - rows_after:,}')

## Step 13: Remove duplicate rows

After the earlier filters, we remove any duplicate rows that still remain. This prevents double-counting in revenue totals and customer analysis. We print the duplicate count and the row counts before and after the drop.

In [ ]:
rows_before = len(df)
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
rows_after = len(df)

print(f'Duplicate rows found: {duplicate_count:,}')
print(f'Rows before: {rows_before:,}')
print(f'Rows after: {rows_after:,}')
print(f'Rows removed: {rows_before - rows_after:,}')

## Step 14: Final data-quality check

This final check confirms that the cleaned dataframe is ready for analysis and Tableau export. We print the final shape, remaining null values, column data types, a numeric summary, and the first 10 cleaned rows so you can inspect the final result quickly.

In [ ]:
print(f'Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('Remaining null values by column:')
print(df.isnull().sum().to_string())
print('
Final data types:')
print(df.dtypes)
print('
Numeric summary:')
print(df.describe().to_string())
print('
First 10 rows of cleaned data:')
print(df.head(10).to_string(index=False))

## Step 15: Save the cleaned dataset

We save the cleaned dataframe to `data/processed/online_retail_cleaned.csv` so the rest of the project can reuse one consistent file. This includes EDA, statistical analysis, Tableau preparation, and any future resubmission or audit checks.

In [ ]:
output_path = Path('data/processed/online_retail_cleaned.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f'Cleaned data saved to {output_path} with {len(df):,} rows.')

## Step 16: Cleaning summary table

| Step | Action | Rows Removed | Reason |
|---|---|---:|---|
| 1 | Rename columns to snake_case | 0 | Standardize column names for easier analysis and scripting |
| 2 | Parse `invoice_date` to datetime | 0 | Enable date-based analysis, sorting, and recency calculations |
| 3 | Drop null `customer_id` rows | 243,007 | Customer-level segmentation needs a valid customer ID |
| 4 | Convert `customer_id` to string and strip `.0` | 0 | Make customer IDs consistent text identifiers |
| 5 | Remove cancellation invoices | 18,744 | Cancelled orders should not count as normal sales |
| 6 | Remove rows where `quantity <= 0` | 0 | No invalid quantity rows remained after cancellations were removed |
| 7 | Remove rows where `unit_price <= 0` | 71 | Exclude free or invalid price rows from revenue analysis |
| 8 | Remove non-product stock codes | 2,853 | Exclude postage, fees, manual lines, and gift control codes |
| 9 | Strip and uppercase text columns | 0 | Standardize `description` and `country` values |
| 10 | Add `revenue` column | 0 | Create the monetary measure used in KPI and segmentation analysis |
| 11 | Remove duplicate rows | 26,055 | Prevent double counting in totals and customer metrics |

**Final cleaned shape:** 776,641 rows x 9 columns